# Spaceship Titanic

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

## Data Read

In [2]:
data_path = "/mnt/e/Kaggle_Competitions/Spaceship_Titanic"
train_dp = os.path.join(data_path, "train.csv")
test_dp = os.path.join(data_path, "test.csv")

df_train = pd.read_csv(train_dp)
df_test = pd.read_csv(test_dp)

In [3]:
df_train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [4]:
df_train.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8514.000000,8512.000000,8510.000000,8485.000000,8510.000000,8505.000000
mean,28.827930,224.687617,458.077203,173.729169,311.138778,304.854791
std,14.489021,666.717663,1611.489240,604.696458,1136.705535,1145.717189
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,47.000000,76.000000,27.000000,59.000000,46.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [5]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


## Data Preprocessing and Feature Engineering

In [6]:
TARGET = "Transported"

# feature engineering
df_train_fe = df_train.copy()
df_test_fe = df_test.copy()
for df in [df_train_fe, df_test_fe]:
    df[['Deck', 'CabinNum', 'Side']] = df['Cabin'].str.split('/', expand=True)
    df['CabinNum'] = pd.to_numeric(df['CabinNum'], errors='coerce')
    df['Group'] = df['PassengerId'].str.split('_').str[0]
    df['MemberID'] = df['PassengerId'].str.split('_').str[1].astype(int)
    df['GroupSize'] = df.groupby('Group')['Group'].transform('count')
    df['IsAlone'] = (df['GroupSize']==1).astype(int)
    
    df['CabinGroup'] = df['CabinNum'] // 100
    df['AgeGroup'] = pd.cut(df['Age'], bins=[0,12,18,25,40,60,100], labels=False)

    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    df['TotalSpend'] = df[spend_cols].sum(axis=1)
    df['NoSpending'] = (df['TotalSpend']==0).astype(int)

    df['Cryo_NoSpend'] = ((df['CryoSleep']=="True") & (df['NoSpending']==1)).astype(int)

    spend_cols = [
        'RoomService', 'FoodCourt', 'ShoppingMall',
        'Spa', 'VRDeck', 'TotalSpend'
    ]
    for col in spend_cols:
        df[col] = np.log1p(df[col])

In [7]:
# miss value process
cat_cols = ['HomePlanet', 'Destination', 'VIP', 'CryoSleep', 'Deck', 'Side']

num_cols = [
    'Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa',
    'VRDeck', 'CabinNum', 'GroupSize', 'MemberID', 'TotalSpend'
]

for col in cat_cols:
    df_train_fe[col] = (df_train_fe[col].fillna('Unknown').astype(str))
    df_test_fe[col] = (df_test_fe[col].fillna('Unknown').astype(str))

for col in num_cols:
    median = df_train_fe[col].median()
    df_train_fe[col] = (df_train_fe[col].fillna(median))
    df_test_fe[col] = (df_test_fe[col].fillna(median))

In [8]:
# Drop columns
drop_cols = ['PassengerId', 'Name', 'Cabin', 'Group']

df_train_drop = df_train_fe.drop(columns=drop_cols)
df_test_drop = df_test_fe.drop(columns=drop_cols)

## Model Train

### Train Setup

In [9]:
# train and test dataset setup
dset_train = df_train_drop.copy()
dset_test = df_test_drop.copy()

X = dset_train.drop(columns=TARGET).copy()
y = dset_train[TARGET].copy()

X_test = dset_test.copy()

X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

In [10]:
# train setup
n_splits = 5

skf = StratifiedKFold(
    n_splits=n_splits, shuffle=True, random_state=42
)

### Model Sur-parameter Selection

In [11]:
from itertools import product
from catboost import CatBoostClassifier

# Parameter search range
param_grid = {
    'depth': [5, 10, 15], 
    'learning_rate': [0.05, 0.1], 
    'l2_leaf_reg': [1, 3],
}

best_score = 0
best_params = None

# Hyperparameter Search
for depth, lr, l2 in product(
    param_grid['depth'], param_grid['learning_rate'], param_grid['l2_leaf_reg']
):
    fold_scores=[]
    print(f'\nDepth={depth}, LR={lr}, L2={l2}')

    for fold,(train_idx,valid_idx) in enumerate(skf.split(X,y)):
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_valid = X.iloc[valid_idx]
        y_valid = y.iloc[valid_idx]

        model = CatBoostClassifier(
            iterations=3000,
            depth=depth,
            learning_rate=lr,
            l2_leaf_reg=l2,
            eval_metric='Accuracy',
            random_seed=42,
            early_stopping_rounds=300,
            verbose=False,
            allow_writing_files=False
        )

        model.fit(
            X_train, y_train, cat_features=cat_cols, 
            eval_set=(X_valid, y_valid)
        )

        pred = model.predict(X_valid)
        acc = accuracy_score(y_valid, pred)
        fold_scores.append(acc)

    mean_acc = np.mean(fold_scores)

    print(f'CV ACC={mean_acc:.5f}')


    if mean_acc > best_score:
        best_score = mean_acc

        best_params = {
            'depth':depth,
            'learning_rate':lr,
            'l2_leaf_reg':l2
        }



print('\nBest Parameters:')
print(best_params)

print(
    f'Best CV = {best_score:.5f}'
)



Depth=5, LR=0.05, L2=1
CV ACC=0.81709

Depth=5, LR=0.05, L2=3
CV ACC=0.81606

Depth=5, LR=0.1, L2=1
CV ACC=0.81778

Depth=5, LR=0.1, L2=3
CV ACC=0.81801

Depth=10, LR=0.05, L2=1
CV ACC=0.81399

Depth=10, LR=0.05, L2=3
CV ACC=0.81571

Depth=10, LR=0.1, L2=1
CV ACC=0.81042

Depth=10, LR=0.1, L2=3
CV ACC=0.81594

Depth=15, LR=0.05, L2=1
CV ACC=0.80697

Depth=15, LR=0.05, L2=3
CV ACC=0.80743

Depth=15, LR=0.1, L2=1
CV ACC=0.80697

Depth=15, LR=0.1, L2=3
CV ACC=0.80778

Best Parameters:
{'depth': 5, 'learning_rate': 0.1, 'l2_leaf_reg': 3}
Best CV = 0.81801


### Final Model Training

In [16]:
final_model=CatBoostClassifier(
    iterations=5000,
    depth=best_params['depth'],
    learning_rate=best_params['learning_rate'],
    l2_leaf_reg=best_params['l2_leaf_reg'],
    random_seed=42,
    verbose=500,
    allow_writing_files=False
)

final_model.fit(X, y, cat_features=cat_cols)

0:	learn: 0.6488572	total: 23.8ms	remaining: 1m 58s
500:	learn: 0.2817829	total: 5.04s	remaining: 45.2s
1000:	learn: 0.2241302	total: 9.3s	remaining: 37.1s
1500:	learn: 0.1861280	total: 13.8s	remaining: 32.1s
2000:	learn: 0.1575088	total: 18.2s	remaining: 27.3s
2500:	learn: 0.1357275	total: 23.2s	remaining: 23.1s
3000:	learn: 0.1184060	total: 29.3s	remaining: 19.5s
3500:	learn: 0.1047576	total: 35.2s	remaining: 15.1s
4000:	learn: 0.0927425	total: 42.2s	remaining: 10.5s
4500:	learn: 0.0831735	total: 47.3s	remaining: 5.25s
4999:	learn: 0.0754287	total: 51.9s	remaining: 0us


CatBoostClassifier(allow_writing_files=False, depth=5, iterations=5000, l2_leaf_reg=3, learning_rate=0.1, random_seed=42, verbose=500)

## Test Result

In [17]:
test_preds = final_model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': df_test['PassengerId'],
    'Transported': test_preds
})

preds_save_path = os.path.join(data_path, 'submission.csv')
submission.to_csv(preds_save_path, index=False)